# 04 — Ingest real open data into MinIO

Lands two public datasets in `s3a://raw-data/` so the medallion-pipeline notebooks (05, 06) have something to chew on.

| Dataset | Format | Size | Source |
|---|---|---|---|
| **MovieLens `ml-latest-small`** | 4 CSVs in one ZIP | ~1 MB | https://grouplens.org/datasets/movielens/ |
| **Wikipedia pageviews — top 1000 articles / day, 7 days** | JSON | ~400 KB total | https://wikimedia.org/api/rest_v1/ |

Both are stable, no API key, safe to re-run.

Run all cells in order. The notebook is idempotent — re-running overwrites the raw objects.

In [ ]:
import io, json, os, tempfile, zipfile
from urllib.request import Request, urlopen

import boto3

s3 = boto3.client(
    "s3",
    endpoint_url=os.environ.get("S3_ENDPOINT", "http://minio:9000"),
    aws_access_key_id=os.environ["MINIO_ACCESS_KEY"],
    aws_secret_access_key=os.environ["MINIO_SECRET_KEY"],
)
BUCKET = "raw-data"
UA = "jupyter-spark-demo/0.1 (demo@example.com)"  # Wikimedia API requires a UA

## MovieLens — download, unzip, upload

In [ ]:
MOVIELENS_URL = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"

print(f"Downloading {MOVIELENS_URL}")
with urlopen(Request(MOVIELENS_URL, headers={"User-Agent": UA})) as r:
    raw = r.read()
print(f"  {len(raw):,} bytes")

wanted = {"links.csv", "movies.csv", "ratings.csv", "tags.csv"}
with zipfile.ZipFile(io.BytesIO(raw)) as zf:
    for name in zf.namelist():
        base = os.path.basename(name)
        if base in wanted:
            payload = zf.read(name)
            key = f"movielens/{base}"
            s3.put_object(Bucket=BUCKET, Key=key, Body=payload, ContentType="text/csv")
            print(f"  uploaded s3a://{BUCKET}/{key}  ({len(payload):,} bytes)")

## Wikipedia pageviews — fetch 7 days, upload each as JSON

In [ ]:
# Fixed historical window so the notebook is deterministic across runs.
from datetime import date, timedelta
DAYS = [date(2025, 1, 1) + timedelta(days=i) for i in range(7)]
PROJECT = "en.wikipedia"

for d in DAYS:
    url = (
        f"https://wikimedia.org/api/rest_v1/metrics/pageviews/top/"
        f"{PROJECT}/all-access/{d.year}/{d.month:02d}/{d.day:02d}"
    )
    with urlopen(Request(url, headers={"User-Agent": UA})) as r:
        payload = r.read()
    key = f"wikipedia/pageviews_{d.isoformat()}.json"
    s3.put_object(Bucket=BUCKET, Key=key, Body=payload, ContentType="application/json")
    # Quick sanity check on the payload.
    summary = json.loads(payload)
    n = len(summary["items"][0]["articles"])
    print(f"  s3a://{BUCKET}/{key}  ({len(payload):,} bytes, {n} articles)")

## Verify what landed

In [ ]:
from itertools import chain
objs = list(chain.from_iterable(
    s3.list_objects_v2(Bucket=BUCKET, Prefix=p).get("Contents", [])
    for p in ("movielens/", "wikipedia/")
))
for o in sorted(objs, key=lambda x: x["Key"]):
    print(f"  {o['Size']:>12,}  s3a://{BUCKET}/{o['Key']}")

Browse the same objects in the MinIO console at http://localhost:9001 (login `minioadmin` / `minioadmin`).

Next: notebook **05** transforms this raw data into a Delta Lake medallion (bronze → silver → gold), and **06** does the same in Iceberg.